In [0]:
import sys
sys.path.append("/Workspace/Shared")
import aws_utils
from pyspark.sql import functions as F

env = aws_utils.get_env_prefix()

SOURCE_TABLE = f"`{env}-cdh`.reference.exchangerates"
TARGET_TABLE = f"`{env}-cdh-delivery`.it.cepalbilive_ref_exchangerates"

business_date_param = dbutils.widgets.get("run_date").strip()

if business_date_param:
    datadate = business_date_param
else:
    datadate = str(
        spark.sql(f"SELECT MAX(BusinessDate) AS bd FROM `{env}-cdh`.primaryactive.exposure")
        .collect()[0].bd
    )



In [0]:
df_result = (
    spark.table(SOURCE_TABLE)
    .filter(F.col("BusinessDate") == datadate)
    .select(
        F.col("BusinessDate").alias("ExchangeRateDate"),
        F.col("OriginalCurrency").alias("CurrencyCode"),
        F.col("ExchangeRate"),
        F.lit(datadate).cast("date").alias("datadate")
    )
)



In [0]:
(
    df_result.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"ExchangeRateDate = '{datadate}'")
    .saveAsTable(TARGET_TABLE)
)